In [ ]:
# =========================================================
# STEP-1 — LOCAL SETUP + W&B + PATHS (OCT LDM)
# =========================================================

import os
import cv2
import glob
import math
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision.utils import make_grid
import torchvision.transforms as T

import wandb  # ✅ ADDED

# =========================================================
# SEEDING (REPRODUCIBILITY)
# =========================================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # ✅ IMPORTANT FIX
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# =========================================================
# W&B INITIALIZATION ✅
# =========================================================
wandb.init(
    project="oct-ldm",   # change if needed
    name="step1_setup",  # run name
    config={
        "image_size": 512,
        "batch_size": None,
        "device": "cuda" if torch.cuda.is_available() else "cpu"
    }
)

print("W&B initialized ✔")

# =========================================================
# LOCAL DATASET PATHS ✅ (CHANGE THIS)
# =========================================================

ROOT = r"D:\Rushi_OCT_Diffusion\CCDS_Split_10K"  # 🔥 CHANGE TO YOUR PATH

TRAIN_DIR = os.path.join(ROOT, "train")
VAL_DIR   = os.path.join(ROOT, "val")
TEST_DIR  = os.path.join(ROOT, "test")

print("\nDataset Paths:")
print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR:", VAL_DIR)
print("TEST_DIR:", TEST_DIR)

# ✅ VALIDATION
assert os.path.exists(TRAIN_DIR), f" TRAIN_DIR not found: {TRAIN_DIR}"
assert os.path.exists(VAL_DIR),   f" VAL_DIR not found: {VAL_DIR}"
assert os.path.exists(TEST_DIR),  f" TEST_DIR not found: {TEST_DIR}"

print("Dataset directories found ✔")

# =========================================================
# OUTPUT ROOT (LOCAL SAVE)
# =========================================================

OUT_ROOT = r"D:\Rushi_OCT_Diffusion\oct_ldm_output"

DIR_SAMPLES     = os.path.join(OUT_ROOT, "samples")
DIR_CHECKPOINTS = os.path.join(OUT_ROOT, "checkpoints")
DIR_PLOTS       = os.path.join(OUT_ROOT, "plots")
DIR_JSON        = os.path.join(OUT_ROOT, "json")
DIR_CSV         = os.path.join(OUT_ROOT, "csv")

for d in [OUT_ROOT, DIR_SAMPLES, DIR_CHECKPOINTS, DIR_PLOTS, DIR_JSON, DIR_CSV]:
    os.makedirs(d, exist_ok=True)

print("Output directories ready ✓")

# =========================================================
# DEVICE
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ✅ LOG DEVICE TO W&B
wandb.config.update({"device_used": str(device)})

# =========================================================
# HELPER — VISUALIZE GRID
# =========================================================

def show_grid(tensor, nrow=8, title=""):
    """
    tensor: [B,1,H,W] or [B,3,H,W]
    """

    grid = make_grid(tensor.clamp(-1, 1), nrow=nrow, normalize=False)
    grid = grid.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=(10, 10))

    if grid.shape[-1] == 1:
        plt.imshow(grid[..., 0], cmap='gray')
    else:
        plt.imshow(grid)

    plt.title(title)
    plt.axis('off')

    # ✅ SAVE IMAGE
    save_path = os.path.join(DIR_PLOTS, f"{title.replace(' ', '_')}.png")
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()

    # ✅ LOG TO W&B
    wandb.log({
        f"grid/{title}": wandb.Image(save_path)
    })

    print(f"Saved + Logged: {save_path}")

# =========================================================
# FINAL CHECK LOG
# =========================================================
wandb.log({
    "status": "step1_complete"
})

print("\n✅ STEP-1 SETUP COMPLETE (LOCAL + W&B READY)")

In [ ]:
# =========================================================
# STEP-2 — OCT Dataset + DataLoader (LOCAL + W&B READY)
# =========================================================

import os
import glob
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import wandb

# =========================================================
# DATASET CLASS (OPTIMIZED + SAFE)
# =========================================================

class OCTDataset(Dataset):

    def __init__(self, root, image_size=512, recursive=True):
        self.root = root
        self.image_size = image_size

        if not os.path.exists(root):
            raise FileNotFoundError(f"Dataset folder not found: {root}")

        pattern = "**/*.png" if recursive else "*.png"
        self.paths = sorted(glob.glob(os.path.join(root, pattern), recursive=recursive))

        if len(self.paths) == 0:
            raise RuntimeError(f"No PNG files found in {root}")

        print(f"Found {len(self.paths)} OCT scans in: {root}")

    def __len__(self):
        return len(self.paths)

    # -----------------------------------------------------
    # LOAD IMAGE
    # -----------------------------------------------------
    def _load_oct_image(self, path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise RuntimeError(f"Failed to load image: {path}")
        return img

    # -----------------------------------------------------
    # ASPECT RATIO PRESERVING RESIZE
    # -----------------------------------------------------
    def _pad_keep_aspect(self, img):
        h, w = img.shape
        target = self.image_size

        scale = min(target / h, target / w)
        nh, nw = int(h * scale), int(w * scale)

        img_resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)

        pad_h = target - nh
        pad_w = target - nw

        top = pad_h // 2
        bottom = pad_h - top
        left = pad_w // 2
        right = pad_w - left

        img_padded = cv2.copyMakeBorder(
            img_resized,
            top, bottom,
            left, right,
            borderType=cv2.BORDER_REFLECT_101
        )

        return img_padded

    # -----------------------------------------------------
    # GET ITEM
    # -----------------------------------------------------
    def __getitem__(self, idx):

        path = self.paths[idx]

        # LOAD
        img_uint8 = self._load_oct_image(path)

        # RESIZE
        img_uint8 = self._pad_keep_aspect(img_uint8)

        # EDGE CONDITION (CANNY)
        edges = cv2.Canny(img_uint8, 50, 150)

        # TO TENSOR
        img = torch.from_numpy(img_uint8).float().unsqueeze(0)
        edges = torch.from_numpy(edges).float().unsqueeze(0)

        # NORMALIZE [-1,1]
        img = (img / 255.0) * 2 - 1
        edges = (edges / 255.0) * 2 - 1

        # CONTIGUOUS (GPU SAFE)
        img = img.contiguous()
        edges = edges.contiguous()

        # METADATA
        patient = os.path.basename(os.path.dirname(path))

        return {
            "image": img,
            "cond": edges,
            "patient": patient,
            "path": path
        }

# =========================================================
# BUILD DATALOADERS (IMPROVED)
# =========================================================

def build_loaders(train_dir, val_dir, test_dir,
                  batch_size=4,
                  num_workers=2,
                  image_size=512):

    train_ds = OCTDataset(train_dir, image_size)
    val_ds   = OCTDataset(val_dir, image_size)
    test_ds  = OCTDataset(test_dir, image_size)

    pin = torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin,
        drop_last=True,
        persistent_workers=(num_workers > 0)
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin,
        drop_last=False,
        persistent_workers=(num_workers > 0)
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin,
        drop_last=False,
        persistent_workers=(num_workers > 0)
    )

    # =========================================================
    # W&B LOGGING (VERY IMPORTANT)
    # =========================================================

    wandb.config.update({
        "train_size": len(train_ds),
        "val_size": len(val_ds),
        "test_size": len(test_ds),
        "batch_size": batch_size,
        "image_size": image_size,
        "num_workers": num_workers
    })

    print("\nDataset Summary:")
    print(f"Train: {len(train_ds)}")
    print(f"Val  : {len(val_ds)}")
    print(f"Test : {len(test_ds)}")

    return train_loader, val_loader, test_loader


# =========================================================
# QUICK VISUALIZATION + W&B CHECK
# =========================================================

def debug_batch(loader, title="debug_batch", save_dir=None):

    batch = next(iter(loader))

    imgs = batch["image"]
    cond = batch["cond"]

    print("Image shape:", imgs.shape)
    print("Cond shape :", cond.shape)

    # TAKE FIRST 8
    imgs = imgs[:8]
    cond = cond[:8]

    # SAVE GRID
    from torchvision.utils import make_grid
    import matplotlib.pyplot as plt

    grid = make_grid(imgs, nrow=4, normalize=True)
    grid = grid.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=(6,6))
    plt.imshow(grid)
    plt.title("Images")
    plt.axis("off")

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        path = os.path.join(save_dir, f"{title}.png")
        plt.savefig(path)
        plt.close()

        # ✅ W&B IMAGE LOG
        wandb.log({
            f"debug/{title}": wandb.Image(path)
        })

        print(f"Saved + Logged batch preview → {path}")
    else:
        plt.show()

In [ ]:
# =========================================================
# STEP-1 — DATASET INSPECTION (LOCAL + W&B + SAVE)
# =========================================================

import os
import glob
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict
import wandb

# =========================================================
# INSPECTION FUNCTION (UPGRADED)
# =========================================================

def inspect_oct_dataset(root, split_name="train", save_csv_dir=None):

    print(f"\nInspecting: {root}")

    if not os.path.exists(root):
        raise FileNotFoundError(f"Folder not found: {root}")

    all_paths = sorted(glob.glob(os.path.join(root, "**/*.png"), recursive=True))

    total_images = len(all_paths)
    print(f"  Total PNG images: {total_images}")

    # -----------------------------------------------------
    # PATIENT DISTRIBUTION
    # -----------------------------------------------------
    patient_counts = defaultdict(int)

    for p in all_paths:
        patient = os.path.basename(os.path.dirname(p))
        patient_counts[patient] += 1

    num_patients = len(patient_counts)
    counts = list(patient_counts.values())

    if num_patients > 0:
        min_imgs = min(counts)
        max_imgs = max(counts)
        avg_imgs = sum(counts) / num_patients

        print(f"  Patients: {num_patients}")
        print(f"  Min imgs per patient: {min_imgs}")
        print(f"  Max imgs per patient: {max_imgs}")
        print(f"  Avg imgs per patient: {avg_imgs:.2f}")
    else:
        print("  No patients detected")
        min_imgs = max_imgs = avg_imgs = 0

    # -----------------------------------------------------
    # RESOLUTION CHECK (MULTIPLE SAMPLES 🔥)
    # -----------------------------------------------------
    resolutions = []

    sample_subset = all_paths[:50]  # check first 50 images
    for p in sample_subset:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            resolutions.append(img.shape)

    if len(resolutions) > 0:
        unique_res = list(set(resolutions))
        print(f"  Unique resolutions (sampled): {unique_res}")
    else:
        print("  Could not read sample images")

    # -----------------------------------------------------
    # SAVE CSV (VERY IMPORTANT 🔥)
    # -----------------------------------------------------
    if save_csv_dir:
        os.makedirs(save_csv_dir, exist_ok=True)

        df = pd.DataFrame({
            "patient_id": list(patient_counts.keys()),
            "num_images": list(patient_counts.values())
        })

        csv_path = os.path.join(save_csv_dir, f"{split_name}_patient_distribution.csv")
        df.to_csv(csv_path, index=False)

        print(f"  Saved CSV → {csv_path}")

        # ✅ W&B TABLE LOG
        wandb.log({
            f"{split_name}_patient_table": wandb.Table(dataframe=df)
        })

    # -----------------------------------------------------
    # W&B LOGGING 🔥
    # -----------------------------------------------------
    wandb.log({
        f"{split_name}/total_images": total_images,
        f"{split_name}/num_patients": num_patients,
        f"{split_name}/min_imgs_per_patient": min_imgs,
        f"{split_name}/max_imgs_per_patient": max_imgs,
        f"{split_name}/avg_imgs_per_patient": avg_imgs
    })

    return num_patients, total_images


# =========================================================
# RUN INSPECTION
# =========================================================

print("===== TRAIN =====")
train_patients, train_images = inspect_oct_dataset(
    TRAIN_DIR, "train", save_csv_dir=DIR_CSV
)

print("\n===== VAL =====")
val_patients, val_images = inspect_oct_dataset(
    VAL_DIR, "val", save_csv_dir=DIR_CSV
)

print("\n===== TEST =====")
test_patients, test_images = inspect_oct_dataset(
    TEST_DIR, "test", save_csv_dir=DIR_CSV
)

# =========================================================
# SUMMARY
# =========================================================

print("\n===== SUMMARY =====")
print(f"TRAIN: {train_patients} patients, {train_images} images")
print(f"VAL:   {val_patients} patients, {val_images} images")
print(f"TEST:  {test_patients} patients, {test_images} images")

# ---------------------------------------------------------
# SAVE SUMMARY CSV 🔥
# ---------------------------------------------------------
summary_df = pd.DataFrame({
    "split": ["train", "val", "test"],
    "patients": [train_patients, val_patients, test_patients],
    "images": [train_images, val_images, test_images]
})

summary_path = os.path.join(DIR_CSV, "dataset_summary.csv")
summary_df.to_csv(summary_path, index=False)

print(f"\nSaved summary CSV → {summary_path}")

# ✅ W&B SUMMARY TABLE
wandb.log({
    "dataset_summary": wandb.Table(dataframe=summary_df)
})

# =========================================================
# SAMPLE IMAGE CHECK (ROBUST)
# =========================================================

sample_files = glob.glob(os.path.join(TRAIN_DIR, "**/*.png"), recursive=True)

if len(sample_files) > 0:
    img = cv2.imread(sample_files[0], cv2.IMREAD_GRAYSCALE)

    if img is not None:
        h, w = img.shape
        print(f"\nSample resolution: ({h}, {w})")

        wandb.log({
            "sample_height": h,
            "sample_width": w
        })
    else:
        print("Failed to load sample image")
else:
    print("No sample images found in TRAIN_DIR")

print("\n✅ DATASET INSPECTION COMPLETE")

In [ ]:
# ============================================================
# VAE (FINAL — STABLE + W&B + PCA READY)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb

# ============================================================
# Safe GroupNorm
# ============================================================

def norm_layer(ch):
    return nn.GroupNorm(min(32, ch), ch)

# ============================================================
# Residual Block
# ============================================================

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()

        self.norm1 = norm_layer(ch)
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1)

        self.norm2 = norm_layer(ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)

        self.act = nn.SiLU()

    def forward(self, x):
        h = self.conv1(self.act(self.norm1(x)))
        h = self.conv2(self.act(self.norm2(h)))
        return x + h

# ============================================================
# Down Block
# ============================================================

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.conv = nn.Conv2d(in_ch, out_ch, 4, 2, 1)
        self.norm = norm_layer(out_ch)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))

# ============================================================
# Up Block
# ============================================================

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.conv = nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1)
        self.norm = norm_layer(out_ch)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))

# ============================================================
# VAE MODEL
# ============================================================

class VAE(nn.Module):
    def __init__(self, im_channels=1, z_channels=32, base_ch=128):
        super().__init__()

        # ---------------- Encoder ----------------
        self.conv_in = nn.Conv2d(im_channels, base_ch, 3, padding=1)

        self.down1 = DownBlock(base_ch, base_ch * 2)      
        self.res1  = ResBlock(base_ch * 2)

        self.down2 = DownBlock(base_ch * 2, base_ch * 4)
        self.res2  = ResBlock(base_ch * 4)

        self.down3 = DownBlock(base_ch * 4, base_ch * 4)
        self.res3  = ResBlock(base_ch * 4)

        mid_ch = base_ch * 4

        # Latent stats
        self.to_stats = nn.Conv2d(mid_ch, z_channels * 2, 3, padding=1)

        # ---------------- Decoder ----------------
        self.from_latent = nn.Conv2d(z_channels, mid_ch, 3, padding=1)

        self.res4 = ResBlock(mid_ch)

        self.up1 = UpBlock(mid_ch, base_ch * 4)
        self.res5 = ResBlock(base_ch * 4)

        self.up2 = UpBlock(base_ch * 4, base_ch * 2)
        self.res6 = ResBlock(base_ch * 2)

        self.up3 = UpBlock(base_ch * 2, base_ch)
        self.res7 = ResBlock(base_ch)

        self.norm_out = norm_layer(base_ch)
        self.conv_out = nn.Conv2d(base_ch, im_channels, 3, padding=1)

    # ============================================================
    # Encode
    # ============================================================

    def encode(self, x):

        x = self.conv_in(x)

        x = self.res1(self.down1(x))
        x = self.res2(self.down2(x))
        x = self.res3(self.down3(x))

        stats = self.to_stats(x)
        mean, logvar = torch.chunk(stats, 2, dim=1)

        logvar = torch.clamp(logvar, -10, 10)  # stability

        return mean, logvar

    # ============================================================
    # Reparameterization
    # ============================================================

    def reparameterize(self, mean, logvar):

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        return mean + eps * std

    # ============================================================
    # Decode
    # ============================================================

    def decode(self, z):

        x = self.from_latent(z)

        x = self.res4(x)

        x = self.res5(self.up1(x))
        x = self.res6(self.up2(x))
        x = self.res7(self.up3(x))

        x = self.conv_out(F.silu(self.norm_out(x)))

        return torch.tanh(x)

    # ============================================================
    # Forward
    # ============================================================

    def forward(self, x):

        mean, logvar = self.encode(x)
        z = self.reparameterize(mean, logvar)
        recon = self.decode(z)

        return recon, mean, logvar, z  # 🔥 added z for PCA

In [ ]:
# ============================================================
# STEP — CONDITIONAL OCT DATASET (FINAL PRO VERSION)
# LOCAL + W&B + DEBUG + SAFE
# ============================================================

import os
import glob
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import wandb

# ============================================================
# DATASET
# ============================================================

class OCTDataset(Dataset):

    def __init__(self, root, image_size=512):

        if not os.path.exists(root):
            raise FileNotFoundError(f"Dataset path not found: {root}")

        self.paths = sorted(
            glob.glob(os.path.join(root, "**/*.png"), recursive=True)
        )

        if len(self.paths) == 0:
            raise RuntimeError(f"No images found in {root}")

        self.image_size = image_size

        print(f"Loaded {len(self.paths)} images from {root}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):

        path = self.paths[idx]

        # -----------------------------------------------------
        # SAFE IMAGE LOAD
        # -----------------------------------------------------
        img_uint8 = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

        if img_uint8 is None:
            raise RuntimeError(f"Failed to load image: {path}")

        # -----------------------------------------------------
        # FIXED RESIZE
        # -----------------------------------------------------
        img_uint8 = cv2.resize(
            img_uint8,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_AREA
        )

        # -----------------------------------------------------
        # EDGE CONDITION
        # -----------------------------------------------------
        edges = cv2.Canny(img_uint8, 50, 150)

        # -----------------------------------------------------
        # TO TENSOR
        # -----------------------------------------------------
        img = torch.from_numpy(img_uint8).float().unsqueeze(0)
        edges = torch.from_numpy(edges).float().unsqueeze(0)

        # -----------------------------------------------------
        # NORMALIZE [-1,1]
        # -----------------------------------------------------
        img = (img / 255.0) * 2 - 1
        edges = (edges / 255.0) * 2 - 1

        # GPU safety
        img = img.contiguous()
        edges = edges.contiguous()

        return {
            "image": img,
            "cond": edges,
            "path": path
        }

# ============================================================
# BUILD LOADERS (LOCAL OPTIMIZED)
# ============================================================

def build_loaders(train_dir, val_dir, test_dir,
                  batch_size=2,
                  num_workers=2,
                  image_size=512):

    train_ds = OCTDataset(train_dir, image_size)
    val_ds   = OCTDataset(val_dir, image_size)
    test_ds  = OCTDataset(test_dir, image_size)

    pin = torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin,
        drop_last=True,
        persistent_workers=(num_workers > 0)
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin,
        drop_last=False,
        persistent_workers=(num_workers > 0)
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin,
        drop_last=False,
        persistent_workers=(num_workers > 0)
    )

    # ============================================================
    # W&B LOGGING
    # ============================================================

    wandb.config.update({
        "train_size": len(train_ds),
        "val_size": len(val_ds),
        "test_size": len(test_ds),
        "batch_size": batch_size,
        "image_size": image_size
    })

    print("\nDataset Summary:")
    print(f"Train: {len(train_ds)}")
    print(f"Val  : {len(val_ds)}")
    print(f"Test : {len(test_ds)}")

    return train_loader, val_loader, test_loader

# ============================================================
# DEBUG + SAVE + W&B VISUALIZATION 🔥
# ============================================================

def debug_and_log_batch(loader, save_dir, name="batch_debug"):

    import matplotlib.pyplot as plt
    from torchvision.utils import make_grid

    os.makedirs(save_dir, exist_ok=True)

    batch = next(iter(loader))

    imgs = batch["image"][:8]
    cond = batch["cond"][:8]

    # ---------------------------------------------------------
    # IMAGE GRID
    # ---------------------------------------------------------
    grid_img = make_grid(imgs, nrow=4, normalize=True)
    grid_img = grid_img.permute(1, 2, 0).cpu().numpy()

    path_img = os.path.join(save_dir, f"{name}_images.png")

    plt.imshow(grid_img)
    plt.title("Images")
    plt.axis("off")
    plt.savefig(path_img, bbox_inches='tight')
    plt.close()

    # ---------------------------------------------------------
    # EDGE GRID
    # ---------------------------------------------------------
    grid_cond = make_grid(cond, nrow=4, normalize=True)
    grid_cond = grid_cond.permute(1, 2, 0).cpu().numpy()

    path_cond = os.path.join(save_dir, f"{name}_edges.png")

    plt.imshow(grid_cond)
    plt.title("Edges")
    plt.axis("off")
    plt.savefig(path_cond, bbox_inches='tight')
    plt.close()

    # ---------------------------------------------------------
    # W&B LOG
    # ---------------------------------------------------------
    wandb.log({
        f"{name}/images": wandb.Image(path_img),
        f"{name}/edges": wandb.Image(path_cond)
    })

    print(f"Saved + Logged → {path_img}")
    print(f"Saved + Logged → {path_cond}")

# ============================================================
# USAGE (LOCAL)
# ============================================================

if __name__ == "__main__":

    wandb.init(project="oct-ldm", name="dataset_loader")

    ROOT = r"D:\Rushi_OCT_Diffusion\CCDS_Split_10K"

    TRAIN_DIR = os.path.join(ROOT, "train")
    VAL_DIR   = os.path.join(ROOT, "val")
    TEST_DIR  = os.path.join(ROOT, "test")

    train_loader, val_loader, test_loader = build_loaders(
        TRAIN_DIR, VAL_DIR, TEST_DIR,
        batch_size=2
    )

    # 🔥 DEBUG CHECK
    debug_and_log_batch(train_loader, save_dir="debug_outputs")

    wandb.finish()

In [ ]:
# ============================================================
# FINAL VAE TRAINING (LOCAL + W&B FULL INTEGRATION 🔥)
# ============================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchvision.utils import make_grid, save_image
import lpips
import pandas as pd
import wandb

# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# W&B LOGIN (RUN ONCE)
# ============================================================
# wandb.login()

# ============================================================
# UTILS
# ============================================================

def norm_layer(ch):
    return nn.GroupNorm(min(32, ch), ch)

# ============================================================
# BLOCKS
# ============================================================

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.norm1 = norm_layer(ch)
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.norm2 = norm_layer(ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.act = nn.SiLU()

    def forward(self, x):
        h = self.conv1(self.act(self.norm1(x)))
        h = self.conv2(self.act(self.norm2(h)))
        return x + h


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 4, 2, 1)
        self.norm = norm_layer(out_ch)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm = norm_layer(out_ch)
        self.act = nn.SiLU()

    def forward(self, x):
        x = self.upsample(x)
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        return x

# ============================================================
# VAE
# ============================================================

class VAE(nn.Module):
    def __init__(self, im_channels=1, z_channels=32, base_ch=128):
        super().__init__()

        self.conv_in = nn.Conv2d(im_channels, base_ch, 3, padding=1)

        self.down1 = DownBlock(base_ch, base_ch * 2)
        self.res1 = ResBlock(base_ch * 2)

        self.down2 = DownBlock(base_ch * 2, base_ch * 4)
        self.res2 = ResBlock(base_ch * 4)

        self.down3 = DownBlock(base_ch * 4, base_ch * 4)
        self.res3 = ResBlock(base_ch * 4)

        mid_ch = base_ch * 4

        self.to_stats = nn.Conv2d(mid_ch, z_channels * 2, 3, padding=1)

        self.from_latent = nn.Conv2d(z_channels, mid_ch, 3, padding=1)

        self.res4 = ResBlock(mid_ch)

        self.up1 = UpBlock(mid_ch, base_ch * 4)
        self.res5 = ResBlock(base_ch * 4)

        self.up2 = UpBlock(base_ch * 4, base_ch * 2)
        self.res6 = ResBlock(base_ch * 2)

        self.up3 = UpBlock(base_ch * 2, base_ch)
        self.res7 = ResBlock(base_ch)

        self.norm_out = norm_layer(base_ch)
        self.conv_out = nn.Conv2d(base_ch, im_channels, 3, padding=1)

    def encode(self, x):
        x = self.conv_in(x)
        x = self.res1(self.down1(x))
        x = self.res2(self.down2(x))
        x = self.res3(self.down3(x))
        stats = self.to_stats(x)
        mean, logvar = torch.chunk(stats, 2, dim=1)
        logvar = torch.clamp(logvar, -10, 10)
        return mean, logvar

    def reparameterize(self, mean, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mean + eps * std

    def decode(self, z):
        x = self.from_latent(z)
        x = self.res4(x)
        x = self.res5(self.up1(x))
        x = self.res6(self.up2(x))
        x = self.res7(self.up3(x))
        x = self.conv_out(F.silu(self.norm_out(x)))
        return torch.tanh(x)

    def forward(self, x):
        mean, logvar = self.encode(x)
        z = self.reparameterize(mean, logvar)
        recon = self.decode(z)
        return recon, mean, logvar

# ============================================================
# LOSSES
# ============================================================

lpips_loss = lpips.LPIPS(net='vgg').to(device)
lpips_loss.eval()
for p in lpips_loss.parameters():
    p.requires_grad = False

def perceptual_loss(pred, target):
    return lpips_loss(pred.repeat(1,3,1,1), target.repeat(1,3,1,1)).mean()

def kl_loss(mean, logvar):
    logvar = torch.clamp(logvar, -30, 20)
    return -0.5 * torch.sum(
        1 + logvar - mean.pow(2) - logvar.exp(),
        dim=[1,2,3]
    ).mean()

def recon_loss(pred, target):
    return F.l1_loss(pred, target)

# ============================================================
# TRAIN FUNCTION
# ============================================================

def train_vae(vae, train_loader, val_loader, epochs=150, lr=5e-5, out_dir="./vae_outputs"):

    os.makedirs(out_dir, exist_ok=True)
    preview_dir = os.path.join(out_dir, "previews")
    os.makedirs(preview_dir, exist_ok=True)

    # 🔥 W&B INIT
    wandb.init(
        project="oct-ldm",
        name="vae_final",
        config={"epochs": epochs, "lr": lr}
    )

    optimizer = torch.optim.AdamW(vae.parameters(), lr=lr)

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type=="cuda"))

    preview = next(iter(val_loader))["image"][:4].to(device)

    best_val = float("inf")
    history = []

    for epoch in range(1, epochs+1):

        beta = min(1e-4, epoch*5e-6)
        perc_weight = 0.005

        vae.train()

        total_l1, total_p, total_k = 0,0,0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):

            img = batch["image"].to(device)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda", enabled=(device.type=="cuda")):
                recon, mean, logvar = vae(img)

                l1 = recon_loss(recon, img)
                p  = perceptual_loss(recon, img)
                k  = kl_loss(mean, logvar)

                loss = l1 + perc_weight*p + beta*k

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            total_l1 += l1.item()
            total_p  += p.item()
            total_k  += k.item()

        total_l1 /= len(train_loader)
        total_p  /= len(train_loader)
        total_k  /= len(train_loader)

        # VALIDATION
        vae.eval()
        val_l1 = 0

        with torch.no_grad():
            for batch in val_loader:
                img = batch["image"].to(device)
                recon, _, _ = vae(img)
                val_l1 += recon_loss(recon, img).item()

        val_l1 /= len(val_loader)

        print(f"\nEpoch {epoch}")
        print(f"L1: {total_l1:.4f} | Val: {val_l1:.4f}")

        # PREVIEW
        with torch.no_grad():
            recon, _, _ = vae(preview)

        vis = torch.cat([preview, recon], dim=0)
        vis = torch.clamp((vis+1)/2,0,1)

        img_path = os.path.join(preview_dir, f"epoch_{epoch}.png")
        save_image(vis, img_path, nrow=4)

        # W&B LOG
        wandb.log({
            "epoch": epoch,
            "train_l1": total_l1,
            "train_perceptual": total_p,
            "train_kl": total_k,
            "val_l1": val_l1,
            "beta": beta,
            "reconstruction": wandb.Image(img_path)
        })

        # SAVE MODEL
        model_path = os.path.join(out_dir, f"vae_epoch_{epoch}.pth")
        torch.save(vae.state_dict(), model_path)
        wandb.save(model_path)

        if val_l1 < best_val:
            best_val = val_l1
            best_path = os.path.join(out_dir, "vae_best.pth")
            torch.save(vae.state_dict(), best_path)
            wandb.save(best_path)
            print("⭐ Best model saved")

        history.append({"epoch":epoch,"train_l1":total_l1,"val_l1":val_l1})

    # SAVE CSV
    df = pd.DataFrame(history)
    csv_path = os.path.join(out_dir, "log.csv")
    df.to_csv(csv_path, index=False)
    wandb.save(csv_path)

    wandb.finish()

    print("✅ Training Complete")

# ============================================================
# RUN
# ============================================================

vae = VAE(z_channels=32, base_ch=128).to(device)

train_vae(vae, train_loader, val_loader)

In [ ]:
# ============================================================
# FINAL PERFECT CONDITIONAL LDM (LOCAL + W&B + RESEARCH READY)
# ============================================================

import os, glob, cv2, copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image
from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler
import lpips
from torchmetrics.image import StructuralSimilarityIndexMeasure
import wandb

# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🚀 Device:", device)

# ============================================================
# LOCAL PATHS (EDIT THIS ONLY)
# ============================================================

ROOT = r"D:\Rushi_OCT_Diffusion\CCDS_Split_10K"
VAE_PATH = r"D:\Rushi_OCT_Diffusion\vae_best.pth"
OUT_DIR = r"D:\Rushi_OCT_Diffusion\FINAL_LDM"

RESUME_CHECKPOINT = os.path.join(OUT_DIR, "ldm_latest.pth")

TRAIN_PATH = os.path.join(ROOT,"train")
VAL_PATH   = os.path.join(ROOT,"val")

PREVIEW_DIR = os.path.join(OUT_DIR,"previews")
BEST_MODEL_PATH = os.path.join(OUT_DIR,"ldm_best.pth")
CHECKPOINT_PATH = os.path.join(OUT_DIR,"ldm_latest.pth")

os.makedirs(PREVIEW_DIR,exist_ok=True)

# ============================================================
# W&B INIT
# ============================================================

wandb.init(
    project="oct-ldm-final",
    name="conditional_ldm_production",
    config={
        "epochs":50,
        "lr":1e-4,
        "batch_size":4,
        "z_channels":32
    }
)

# ============================================================
# DATASET
# ============================================================

class OCTDataset(Dataset):
    def __init__(self,root,size=512):
        self.paths = sorted(glob.glob(os.path.join(root,"**/*.png"),recursive=True))
        self.size=size
        print(f"{len(self.paths)} images from {root}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self,i):
        img = cv2.imread(self.paths[i],cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img,(self.size,self.size))
        img = torch.from_numpy(img).float()/255.0
        img = img.unsqueeze(0)*2-1
        return {"image":img}

train_loader = DataLoader(OCTDataset(TRAIN_PATH),batch_size=4,shuffle=True,num_workers=2)
val_loader   = DataLoader(OCTDataset(VAL_PATH),batch_size=4,shuffle=False,num_workers=2)

# ============================================================
# VAE (CORRECT ARCH)
# ============================================================

def norm_layer(ch):
    return nn.GroupNorm(min(32,ch),ch)

class ResBlock(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.norm1 = norm_layer(ch)
        self.conv1 = nn.Conv2d(ch,ch,3,padding=1)
        self.norm2 = norm_layer(ch)
        self.conv2 = nn.Conv2d(ch,ch,3,padding=1)
        self.act = nn.SiLU()

    def forward(self,x):
        h = self.conv1(self.act(self.norm1(x)))
        h = self.conv2(self.act(self.norm2(h)))
        return x+h

class DownBlock(nn.Module):
    def __init__(self,in_ch,out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch,out_ch,4,2,1)
        self.norm = norm_layer(out_ch)
        self.act  = nn.SiLU()

    def forward(self,x):
        return self.act(self.norm(self.conv(x)))

class UpBlock(nn.Module):
    def __init__(self,in_ch,out_ch):
        super().__init__()
        self.conv = nn.ConvTranspose2d(in_ch,out_ch,4,2,1)
        self.norm = norm_layer(out_ch)
        self.act  = nn.SiLU()

    def forward(self,x):
        return self.act(self.norm(self.conv(x)))

class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        b=128; z=32

        self.conv_in = nn.Conv2d(1,b,3,1,1)
        self.down1 = DownBlock(b,b*2); self.res1 = ResBlock(b*2)
        self.down2 = DownBlock(b*2,b*4); self.res2 = ResBlock(b*4)
        self.down3 = DownBlock(b*4,b*4); self.res3 = ResBlock(b*4)

        self.to_stats = nn.Conv2d(b*4,z*2,3,1,1)
        self.from_latent = nn.Conv2d(z,b*4,3,1,1)

        self.res4 = ResBlock(b*4)
        self.up1 = UpBlock(b*4,b*4); self.res5 = ResBlock(b*4)
        self.up2 = UpBlock(b*4,b*2); self.res6 = ResBlock(b*2)
        self.up3 = UpBlock(b*2,b);   self.res7 = ResBlock(b)

        self.norm_out = norm_layer(b)
        self.conv_out = nn.Conv2d(b,1,3,1,1)

    def encode(self,x):
        x = self.conv_in(x)
        x = self.res1(self.down1(x))
        x = self.res2(self.down2(x))
        x = self.res3(self.down3(x))
        mean,logvar = torch.chunk(self.to_stats(x),2,1)
        return mean,logvar.clamp(-10,10)

    def decode(self,z):
        x = self.from_latent(z)
        x = self.res4(x)
        x = self.res5(self.up1(x))
        x = self.res6(self.up2(x))
        x = self.res7(self.up3(x))
        return torch.tanh(self.conv_out(F.silu(self.norm_out(x))))

vae = VAE().to(device)
vae.load_state_dict(torch.load(VAE_PATH,map_location=device))
vae.eval().requires_grad_(False)

# ============================================================
# CONDITION (IMPROVED)
# ============================================================

sobel_x = torch.tensor([[1,0,-1],[2,0,-2],[1,0,-1]],device=device).view(1,1,3,3).float()
sobel_y = torch.tensor([[1,2,1],[0,0,0],[-1,-2,-1]],device=device).view(1,1,3,3).float()

def get_condition(img):
    img=(img+1)/2
    img = F.avg_pool2d(img,3,stride=1,padding=1)
    gx = F.conv2d(img,sobel_x,padding=1)
    gy = F.conv2d(img,sobel_y,padding=1)
    edges = torch.sqrt(gx**2+gy**2)
    edges = edges/(edges.amax(dim=[1,2,3],keepdim=True)+1e-8)
    edges = edges*2-1
    return F.interpolate(edges,size=(64,64))

# ============================================================
# UNET + EMA
# ============================================================

unet = UNet2DModel(
    sample_size=64,
    in_channels=33,
    out_channels=32,
    layers_per_block=2,
    block_out_channels=(128,256,512,512),
    down_block_types=("DownBlock2D","AttnDownBlock2D","AttnDownBlock2D","DownBlock2D"),
    up_block_types=("UpBlock2D","AttnUpBlock2D","AttnUpBlock2D","UpBlock2D")
).to(device)

ema_unet = copy.deepcopy(unet)
for p in ema_unet.parameters(): p.requires_grad_(False)

def update_ema(decay=0.999):
    with torch.no_grad():
        for e,p in zip(ema_unet.parameters(),unet.parameters()):
            e.data.mul_(decay).add_(p.data,alpha=1-decay)

# ============================================================
# SCHEDULERS
# ============================================================

scheduler = DDPMScheduler(num_train_timesteps=1000,beta_schedule="squaredcos_cap_v2")
ddim = DDIMScheduler(num_train_timesteps=1000,beta_schedule="squaredcos_cap_v2")

optimizer = torch.optim.AdamW(unet.parameters(),lr=1e-4)
scaler = torch.amp.GradScaler(enabled=(device.type=="cuda"))

# ============================================================
# METRICS
# ============================================================

lpips_metric = lpips.LPIPS(net="vgg").to(device).eval()
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

# ============================================================
# GENERATE
# ============================================================

@torch.no_grad()
def generate(img):
    ema_unet.eval()
    mean,_ = vae.encode(img)
    cond = get_condition(img)
    z = torch.randn_like(mean)
    ddim.set_timesteps(100)

    for t in ddim.timesteps:
        xt_cond = torch.cat([z,cond],dim=1)
        noise_pred = ema_unet(xt_cond,t).sample
        z = ddim.step(noise_pred,t,z).prev_sample

    return vae.decode(z)

# ============================================================
# VALIDATE
# ============================================================

@torch.no_grad()
def validate():
    total_ssim,total_lpips,total_psnr,count = 0,0,0,0

    for i,batch in enumerate(val_loader):
        if i>5: break

        img=batch["image"].to(device)
        gen=generate(img)

        real=(img+1)/2
        gen=(gen+1)/2

        total_ssim+=ssim_metric(gen,real).item()

        total_lpips+=lpips_metric(
            gen.repeat(1,3,1,1),
            real.repeat(1,3,1,1)
        ).mean().item()

        mse=F.mse_loss(gen,real)
        total_psnr+=(10*torch.log10(1.0/mse)).item()

        count+=1

    return total_ssim/count,total_lpips/count,total_psnr/count

# ============================================================
# TRAIN
# ============================================================

start_epoch=1
best_ssim=0

if os.path.exists(RESUME_CHECKPOINT):
    ck=torch.load(RESUME_CHECKPOINT,map_location=device)
    unet.load_state_dict(ck["model"])
    ema_unet.load_state_dict(ck["ema_model"])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    start_epoch=ck["epoch"]+1
    best_ssim=ck.get("best_val",0)

for epoch in range(start_epoch,51):

    unet.train()
    total=0

    for batch in tqdm(train_loader,desc=f"Epoch {epoch}"):

        img=batch["image"].to(device)

        with torch.no_grad():
            mean,_ = vae.encode(img)
            z = mean

        cond=get_condition(img)
        noise=torch.randn_like(z)
        t=torch.randint(0,1000,(z.size(0),),device=device)

        xt=scheduler.add_noise(z,noise,t)
        xt_cond=torch.cat([xt,cond],dim=1)

        with torch.autocast(device_type=device.type,enabled=(device.type=="cuda")):
            pred=unet(xt_cond,t).sample
            loss=F.mse_loss(pred,noise)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(),1.0)
        scaler.step(optimizer)
        scaler.update()

        update_ema()
        total+=loss.item()

    train_loss=total/len(train_loader)

    val_ssim,val_lpips,val_psnr=validate()

    # ========================================================
    # SAVE + W&B LOG
    # ========================================================

    preview = generate(next(iter(val_loader))["image"][:4].to(device))
    preview = (preview+1)/2
    img_path = f"{PREVIEW_DIR}/epoch_{epoch}.png"
    save_image(preview,img_path,nrow=2)

    wandb.log({
        "epoch":epoch,
        "train_loss":train_loss,
        "SSIM":val_ssim,
        "LPIPS":val_lpips,
        "PSNR":val_psnr,
        "preview": wandb.Image(img_path)
    })

    if val_ssim>best_ssim:
        best_ssim=val_ssim
        torch.save(ema_unet.state_dict(),BEST_MODEL_PATH)
        wandb.save(BEST_MODEL_PATH)

    torch.save({
        "epoch":epoch,
        "model":unet.state_dict(),
        "ema_model":ema_unet.state_dict(),
        "optimizer":optimizer.state_dict(),
        "scaler":scaler.state_dict(),
        "best_val":best_ssim
    },CHECKPOINT_PATH)

    print(f"\nEpoch {epoch} | Loss: {train_loss:.4f} | SSIM: {val_ssim:.4f}")

wandb.finish()

print("✅ TRAINING COMPLETE")